Binning and Binarization

In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [6]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

In [7]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [8]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [9]:
X_train , X_test , y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [10]:
model = DecisionTreeClassifier()
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test,y_pred)

0.6368715083798883

In [11]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6666916354556803)

Applying Discritizer

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


age_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('kbins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile'))
])

fare_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('kbins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile'))
])

trf = ColumnTransformer([
    ('age', age_pipe, [0]),
    ('fare', fare_pipe, [1])
])

X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


In [22]:
trf.named_transformers_['age'][1].bin_edges_

array([array([ 0.42, 21.  , 28.  , 38.  , 80.  ])], dtype=object)

In [23]:
output = pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:,0],
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:,1]
})

In [24]:
output.head()

,age,age_trf,fare,fare_trf
331,45.5,3.0,28.5000,3.0
733,23.0,1.0,13.0000,2.0
382,32.0,2.0,7.9250,1.0
704,26.0,1.0,7.8542,0.0
813,6.0,0.0,31.2750,3.0


In [26]:
output['age_labels'] = pd.cut(x=X_train['Age'],
                                    bins=trf.named_transformers_['age'][1].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=X_train['Fare'],
                                    bins=trf.named_transformers_['fare'][1].bin_edges_[0].tolist())

In [28]:
output.head()

,age,age_trf,fare,fare_trf,age_labels,fare_labels
331,45.5,3.0,28.5000,3.0,"(38.0, 80.0]","(21.045, 39.688]"
733,23.0,1.0,13.0000,2.0,"(21.0, 28.0]","(10.5, 21.045]"
382,32.0,2.0,7.9250,1.0,"(28.0, 38.0]","(7.889, 10.5]"
704,26.0,1.0,7.8542,0.0,"(21.0, 28.0]","(0.0, 7.889]"
813,6.0,0.0,31.2750,3.0,"(0.42, 21.0]","(21.045, 39.688]"


In [29]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

In [30]:
accuracy_score(y_test,y_pred2)

0.6983240223463687

In [31]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(


np.float64(0.6588389513108613)